In [ ]:
%pip uninstall -y transformers tokenizers vllm
%pip install -U "transformers>=4.55.2,<5" "tokenizers>=0.21.1,<0.22" "vllm==0.11.0"

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import os
import sys
import typing as tp
import numpy as np
import subprocess
import asyncio
import json
import time
import aiohttp
import requests
import threading
import logging
import select

import pprint
from pathlib import Path
from datetime import datetime, timezone

from IPython.display import HTML, display

### Модели, которые проверенно заработают на T4

`Qwen/Qwen3-4B-Thinking-2507`

`Qwen/Qwen3-4B-Instruct-2507`

`Qwen/Qwen3-8B-AWQ`


In [ ]:
def watch(process) -> None:
    process_finished = False
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            formatted_time = datetime.now().strftime("'%Y-%m-%d %H:%M:%S UTC")
            display(HTML(f"<p style='color: cyan;'>VLLM - {formatted_time} - {line}</p>"))


process = subprocess.Popen("vllm serve --host=0.0.0.0 --port=9999 --model=Qwen/Qwen3-4B-Instruct-2507 --max-model-len=24000 --dtype=float16 --reasoning-parser qwen3".split(' '),
                           stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT,
                           text=True,
                           bufsize=-1)


logs_watcher = threading.Thread(target=watch, args=(process, ))
logs_watcher.start()

for _ in range(300):
    if not logs_watcher.is_alive():
        print("Watcher finished")
        break
    try:
        response = requests.get('http://localhost:9999/health')
        if response.status_code == 200:
            print("Readiness probe completed!")
            break
    except Exception as e:
        pass
    time.sleep(1)



In [ ]:
async def get_models() -> None:
    async with aiohttp.ClientSession() as sess:
        async with sess.get('http://localhost:9999/v1/models') as resp:
            return await resp.json()

data = await get_models()
pprint.pprint(data)
MODEL_TO_USE = data['data'][0]['id']
print(f"{MODEL_TO_USE=}")

# Попробуем поговорить с моделью о чем-то простом

In [ ]:
async def infer_dialog_insta(model: str, messages: list[dict[str, str]]) -> tuple[str, str]:
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": 2048,  # Maximum number of tokens to generate
        # "temperature": 0.7  # Controls the randomness of the output
    }

    async with aiohttp.ClientSession() as session:
        async with session.post(
            "http://localhost:9999/v1/chat/completions",
            headers={"Content-Type": "application/json"},
            json=payload,
        ) as resp:
            result = await resp.json()
            if resp.status != 200:
                return json.dumps(result), False

        # pprint.pprint(result)
        response = result["choices"][0]["message"]["content"]
        return response, True


messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
]

async def ask_insta(text: str) -> None:
    messages.append({
        "role": "user",
        "content": text
    })

    answer, status = await infer_dialog_insta(MODEL_TO_USE, messages)
    display(HTML("<p style='color: red;'>ASSISTANT:</p>"))
    print(answer)
    messages.append({
        "role": "assistant",
        "content": answer
    })

In [ ]:
await ask_insta('Привет! Кто ты?')

In [ ]:
await ask_insta('Расскажи, как складывать числа в столбик')

In [ ]:
await ask_insta('Переведи ответ на английский и сократи его, постарайся обойтись в 2 абзаца')

# Для интерактивных приложений есть возможность организовать стриминговое API

In [ ]:
messages=[]

In [ ]:
async def infer_dialog_stream(model: str, messages: list[dict[str, str]]):
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": 2048,  # Maximum number of tokens to generate
        # "temperature": 0.7  # Controls the randomness of the output
        "stream": True,
    }
    async with aiohttp.ClientSession() as session:
        async with session.post(
            "http://localhost:9999/v1/chat/completions",
            headers={"Content-Type": "application/json"},
            json=payload,
        ) as resp:
            if resp.status != 200:
                yield None, False

            is_reasoning = False
            async for line in resp.content:
                line = line.decode('utf-8')
                if not line.startswith('data: '):
                    continue
                if line[6:] == '[DONE]\n':
                    break
                try:
                    content = json.loads(line[6:])
                    delta = content['choices'][0]['delta']
                    if 'reasoning_content' in delta and delta['reasoning_content'] != '':
                        if not is_reasoning:
                            is_reasoning = True
                            yield "<think>\n", True
                        yield delta['reasoning_content'], True
                    elif 'content' in delta and delta['content'] != '':
                        if is_reasoning:
                            is_reasoning = False
                            yield "</think>\n", True
                        yield delta['content'], True

                except json.JSONDecodeError:
                    print(f'error decode: ({line})')
                    yield 'ERR', True



async def reset_context():
    global messages
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. You absolutely love teaching students new things."
        },
    ]

async def ask_stream(text: str) -> None:
    messages.append({
        "role": "user",
        "content": text
    })

    display(HTML("<p style='color: red;'>ASSISTANT:</p>"))

    answer = ''
    async for token, status in infer_dialog_stream(MODEL_TO_USE, messages):
        if not status:
            break
        answer += token
        print(token, end='')
    print()

    messages.append({
        "role": "assistant",
        "content": answer
    })


In [ ]:
await reset_context()

In [ ]:
await ask_stream("Расскажи анекдот")

In [ ]:
await ask_stream("Добавь сюда еще армянское радио")

# Попробуем поиграться в замеры скорости

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer(action: str) -> tp.Generator[None, None, None]:
    start = time.monotonic()
    yield
    stop = time.monotonic()
    print(f"Duration [{action}]: {(stop - start) * 1000:.0f}ms")

In [ ]:
messages_of_first_request = [
    {
        "role": "system",
        "content": "You are a helpful assistant. You absolutely love teaching students new things."
    },
    {
        "role": "user",
        "content": "Научи меня, как правильно подавать подачу в волейболе. Будь достаточно краток."
    },
]

for i in range(4):
    with timer(f"Approach {i}"):
        await infer_dialog_insta(MODEL_TO_USE, messages_of_first_request)

# А теперь с тяжелой стадией PREFILL

In [ ]:
TEXT = """
Путь к звёздам: подготовка первого космонавта
Историческое событие 12 апреля 1961 года, когда Юрий Гагарин совершил первый в истории человечества космический полёт, был результатом интенсивной и всесторонней подготовки, продолжавшейся несколько лет. Этот подвиг стал возможен благодаря комплексной системе обучения, включавшей теоретические знания, психологическую закалку и физическую подготовку, которые сформировали идеального космонавта.​

Теоретическое образание и изучение техники
Первоначальный этап подготовки был посвящён масштабному обучению. Военные лётчики, отобранные для космической программы, прошли через серьезные лекционные занятия, изучая теорию космического полёта, основы астрономии и физики. Кандидатов детально знакомили с устройством космического корабля «Восток», изучали организационные аспекты предстоящей миссии. Все полученные знания проверялись на экзаменах — будущий пилот должен был знать всё досконально. Эта фундаментальная база была критически важна для понимания всех аспектов предстоящего полёта.​

Психологическая закалка
Психологическая подготовка представляла собой одно из самых сложных испытаний. Кандидатов помещали в специальную сурдобарокамеру, где они находились в изоляции в течение десяти дней в условиях пониженного атмосферного давления и повышенного содержания кислорода. В полной тишине, в ограниченном пространстве Гагарин мог лишь выполнять зарядку, читать и вести дневник. Такие испытания проверяли психологическую устойчивость и способность переносить стресс в экстремальных условиях.​

Интенсивная физическая подготовка
Физическая подготовка была одним из ключевых элементов программы. Гагарин, как и другие кандидаты, находился в идеальной физической форме. На ежедневных тренировках особенное внимание уделялось укреплению мышц груди и брюшного пресса, поскольку во время взлёта и посадки космонавт испытывал давление, в шесть-восемь раз превышающее вес его тела. Неподготовленный организм в таких условиях не мог бы функционировать нормально.​

Программа включала разнообразные упражнения на координацию: гимнастику, плавание, прыжки на батуте, занятия на перекладине и брусьях. Гагарин должен был в совершенстве чувствовать своё тело и контролировать все движения, что было особенно критично в условиях невесомости. Каждая тренировка проводилась под пристальным наблюдением врачей и специалистов.​

Моделирование условий невесомости
Одним из самых инновационных аспектов подготовки была тренировка в условиях невесомости. Для этого использовался самолёт Ту-104, в котором оборудовали специальное пространство, покрытое мягкими матами. Самолёт взлетал на высоту 10 километров, а затем отключал двигатели и падал в течение 35-40 секунд. Это время невесомости было использовано для отработки упражнений на координацию, практики приёма пищи и записи наблюдений. Такие полёты позволяли космонавтам приобрести практический опыт и преодолеть психологический барьер перед полётом.​

Финальный этап: подготовка в скафандре
На заключительных этапах подготовки упражнения выполнялись в тяжёлом скафандре, причём практически прикованным к креслу. Это позволяло отработать самые жёсткие условия, максимально приближённые к реальному полёту. Утром перед историческим стартом Гагарин прошёл последнюю предстартовую подготовку, врачи проверили его самочувствие. По собственному признанию космонавта, он чувствовал себя хорошо, был выспавшимся и отдохнувшим.​

Выбор первого космонавта
Среди всех кандидатов выделились три лидера: Юрий Гагарин, Григорий Нелюбов и Герман Титов. Хотя Титов был подготовлен несколько лучше, выбор пал на Гагарина. Это решение оказалось судьбоносным для истории космонавтики. Полёт, длившийся 108 минут, вывел Восток-1 на орбиту, облетев вокруг Земли на высоте 302 километра.​

Подготовка Юрия Гагарина к первому космическому полёту — это уникальный пример того, как комплексный подход к обучению, сочетающий теорию, практику, психологическую подготовку и физический тренинг, позволил человечеству сделать гигантский скачок в освоении космоса. Благодаря преданности, профессионализму и мужеству Гагарина и его наставников, этот день остался в истории как символ человеческих достижений.
"""

questions = [
    "Какой самолет использовался в испытаниях?",
    "Кто, кроме Гагарина, был кандидатом на первый полет?",
    "Говорится ли в тексте про устройство космического корабля, на котором был совершен полет?",
    "Какова была физическая форма Гагарина перед активной подготовкой?",
]


messages_of_second_request = [
]

for question in questions:
    dialog = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": TEXT
        },
        {
            "role": "user",
            "content": f"Максимально кратко ответь на вопрос по тексту: {question}"
        },
    ]
    messages_of_second_request.append(dialog)

for index, dialog in enumerate(messages_of_second_request):
    with timer(f"Question {index}"):
        print(dialog[-1]['content'])
        print(await infer_dialog_insta(MODEL_TO_USE, dialog))

# Ну и теперь инференс в параллель

In [ ]:
tasks = []
for index, dialog in enumerate(messages_of_second_request):
    tasks.append(infer_dialog_insta(MODEL_TO_USE, dialog))

with timer("Joined inputs"):
    await asyncio.gather(*tasks)

In [ ]:
tasks = []
for index, dialog in enumerate(messages_of_second_request):
    tasks.append(infer_dialog_insta(MODEL_TO_USE, dialog))

with timer("Joined inputs"):
    await asyncio.gather(*tasks)